In [1]:
import polars as pl
pl.Config.set_fmt_str_lengths(200)

import requests as rq
import json

### Keys

In [2]:
# Get demo API key
def get_demo_key():
    f = open("/home/vikas/Documents/CG_demo_key.json")
    key_dict = json.load(f)
    return key_dict["key"]

In [3]:
# Get pro API key
def get_pro_key():
    f = open("/home/vikas/Documents/CG_pro_key.json")
    key_dict = json.load(f)
    return key_dict["key"]

### API status

In [8]:
PUB_URL = "https://api.coingecko.com/api/v3"
PRO_URL = "https://pro-api.coingecko.com/api/v3"

In [9]:
def get_response(endpoint, headers, params, URL):
    url = "".join((URL, endpoint))
    response = rq.get(url, headers = headers, params = params)
    if response.status_code == 200:
        data = response.json()
        return data
    else:
        print(f"Failed to fetch data, check status code {response.status_code}")

In [10]:
use_demo = {
           "accept": "application/json",
           "x-cg-demo-api-key" : get_demo_key() 
}

In [11]:
get_response("/ping", use_demo, "", PUB_URL)

{'gecko_says': '(V3) To the Moon!'}

In [37]:
get_pro_key()

'CG-kP6pGVvVp8cKvMwhv5k9Nfc7'

#### List of coins

In [12]:
def get_list_of_coins():

    response = get_response("/coins/list", use_demo, "", PUB_URL)

    return pl.DataFrame(response) 

In [13]:
df_coins = get_list_of_coins()

df_coins.filter(pl.col("symbol") == "eth")

id,symbol,name
str,str,str
"""bifrost-bridged-eth-bifrost""","""eth""","""Bifrost Bridged ETH (Bifrost)"""
"""bridged-binance-peg-ethereum-opbnb""","""eth""","""Bridged Binance-Peg Ethereum (opBNB)"""
"""bridged-wrapped-ether-eclipse""","""eth""","""Bridged Wrapped Ether (Eclipse)"""
"""bridged-wrapped-ether-starkgate""","""eth""","""Bridged Ether (StarkGate)"""
"""ethereum""","""eth""","""Ethereum"""
…,…,…
"""laika-bridged-eth-laika""","""eth""","""Laika Bridged ETH (Laika)"""
"""nova-eth""","""eth""","""Nova Merged ETH (zkLink)"""
"""osmosis-alleth""","""eth""","""Osmosis allETH"""


### OHLC

In [14]:
def get_ohlc_data(coin_id, target_curr, days, precision):

    ohlc_params = {
            "vs_currency": target_curr,
            "days": days,
            "precision": precision
    }

    ohlc_list_response = get_response(f"/coins/{coin_id}/ohlc",
                                      use_demo,
                                      ohlc_params,
                                      PUB_URL)

    df_ohlc = pl.DataFrame(ohlc_list_response, 
                           schema = ["timestamp", "open", "high", "low", "close"],
                           orient = "row")

    df_ohlc = df_ohlc.with_columns(
                  pl.from_epoch(pl.col("timestamp"),
                                time_unit = "ms").alias("timestamp")
         )

    return df_ohlc

In [22]:
get_ohlc_data("ethereum", "usd", "365", "2")

timestamp,open,high,low,close
datetime[ms],f64,f64,f64,f64
2024-07-24 00:00:00,3437.6,3534.98,3403.72,3482.98
2024-07-28 00:00:00,3481.46,3484.66,3098.69,3254.61
2024-08-01 00:00:00,3243.83,3395.72,3202.15,3233.08
2024-08-05 00:00:00,3229.41,3240.67,2655.59,2682.44
2024-08-09 00:00:00,2690.65,2709.43,2171.25,2685.01
…,…,…,…,…
2025-07-07 00:00:00,2572.33,2630.57,2478.63,2571.36
2025-07-11 00:00:00,2570.82,2979.89,2521.06,2948.45
2025-07-15 00:00:00,2948.78,3074.13,2908.75,3012.18


#### Verify column assignment

In [19]:
def validate_columns(df):    

    # Create expressions for max and min across the OHLC columns
    validation = df.with_columns([
        pl.max_horizontal(["open", "high", "low", "close"]).alias("row_max"),
        pl.min_horizontal(["open", "high", "low", "close"]).alias("row_min")
    ])

    not_valid = (
        (validation["high"] != validation["row_max"]) &
        (validation["low"] != validation["row_min"])
    ).all()
    
    print("Invalid OHLC rows:", not_valid)

    return None

In [21]:
validate_columns(get_ohlc_data("ethereum", "usd", "365", "2"))

Invalid OHLC rows: False


#### Export to CSV / XLSX

In [32]:
df_export = get_ohlc_data("bitcoin", "eur", "14", "2")

df_export.write_csv("/home/vikas/Desktop/GitHub/CoinGecko_export_data/BTC_EUR_export.csv")
df_export.write_excel("/home/vikas/Desktop/GitHub/CoinGecko_export_data/BTC_EUR_export.xlsx",
                      autofit = True)

In [33]:
df_export.write_excel("/home/vikas/Desktop/GitHub/CoinGecko_export_data/BTC_EUR_export_B5.xlsx",
                      autofit = True,
                      position = "B5")

In [35]:
df_export.write_excel("/home/vikas/Desktop/GitHub/CoinGecko_export_data/BTC_EUR_export_date_format.xlsx",
                      autofit = True,
                      position = "A5",
                      column_formats = {"timestamp" : "dd/mm/yyyy"})

### OHLCV

#### Get top pools for a specific network

In [25]:
use_pro = {
         "accept": "application/json",
         "x-cg-pro-api-key" : get_pro_key()
}

In [26]:
def get_url(url_type,
            network,
            dex = "",
            pool_address = "",
            token_address = ""):   

    url_dict = {
        "trending_pools": f"/onchain/networks/{network}/trending_pools",
        "top_pools": f"/onchain/networks/{network}/pools",
        "top_pools_dex": f"/onchain/networks/{network}/dexes/{dex}/pools",
        "specific_pool_dex": f"/onchain/networks/{network}/pools/{pool_address}",
        "top_pools_add": f"/onchain/networks/{network}/tokens/{token_address}/pools"
    }

    return url_dict[url_type]

def collect_response(list_response):    

    response_all = []

    for response in list_response["data"]:
        all_attributes = response["attributes"]
        daily_tx = all_attributes["transactions"]["h24"]
        rel = response["relationships"]
        
        temp_dict = dict(
            pair = all_attributes["name"],
            dex = rel["dex"]["data"]["id"],
            add = all_attributes["address"],
            fdv_usd = all_attributes["fdv_usd"],
            market_cap_usd = all_attributes["market_cap_usd"],
            daily_volume = all_attributes["volume_usd"]["h24"],
            daily_price_change = all_attributes["price_change_percentage"]["h24"],
            daily_buys = daily_tx["buys"],
            daily_sells = daily_tx["sells"],
            daily_buyers = daily_tx["buyers"],
            daily_sellers = daily_tx["sellers"]
        )
        
        response_all.append(temp_dict)

    return response_all

def get_top_pools_network(network, sort_by_col):

    target_url = get_url("top_pools", network)

    toppool_list_response = get_response(target_url,
                                         use_pro, 
                                         "",
                                         PRO_URL)

    toppool_all = collect_response(toppool_list_response)   

    return pl.DataFrame(toppool_all).sort([f"{sort_by_col}"], descending = True)

In [27]:
get_top_pools_network("eth", "daily_volume").limit(5)

pair,dex,add,fdv_usd,market_cap_usd,daily_volume,daily_price_change,daily_buys,daily_sells,daily_buyers,daily_sellers
str,str,str,str,str,str,str,i64,i64,i64,i64
"""USDC / WETH 0.01%""","""uniswap_v3""","""0xe0554a476a092703abdb3ef35c80e0d76d32939f""","""41178587532.5681""","""64237283497.7272""","""98446660.3835193""","""-0.29""",9695,6878,5147,2658
"""WETH / USDC 0.05%""","""uniswap_v3""","""0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640""","""8360156535.84822""","""8365587789.80868""","""90274148.1976303""","""2.77""",2515,3483,1067,2100
"""VINEAI / WETH""","""uniswap_v2""","""0xa5179ef1e9d7c894041b17299c238efd9712fb3e""","""659477.890923149""",null,"""8323311.56033265""","""8584.09""",1696,846,585,309
"""VOLT / WETH""","""uniswap_v2""","""0x33b804e9a60da9853a0326a3218b786c6295fb18""","""6532760.18481066""",null,"""7483298.6521417""","""67.26""",1476,577,701,361
"""JOY / WETH""","""uniswap_v2""","""0x780b291337bd55e7ed712f17feffd8cdffbce11b""","""167470.546094266""",null,"""737051.912744664""","""3630.83""",992,921,541,410


#### OHLCV for a specific pool

In [29]:
def get_ohlcv_data(network, pool_add, timeframe):

    ohlcv_params = ""

    ohlcv_list_response = get_response(f"/onchain/networks/{network}/pools/{pool_address}/ohlcv/{timeframe}",
                                       use_pro,
                                       ohlcv_params,
                                       PRO_URL)

    df_ohlcv = pl.DataFrame(ohlcv_list_response["data"]["attributes"]["ohlcv_list"], 
                           schema = ["timestamp", 
                                     "open", 
                                     "high",
                                     "low",
                                     "close",
                                     "volume"],
                           orient = "row")

    df_ohlcv = df_ohlcv.with_columns(
                  pl.from_epoch(pl.col("timestamp"),
                                time_unit = "s").alias("timestamp")
         )

    return df_ohlcv

In [30]:
network = "eth"
pool_address = "0xe0554a476a092703abdb3ef35c80e0d76d32939f"
timeframe = "minute?aggregate=5"

df_ohlcv = get_ohlcv_data(network, pool_address, timeframe)

df_ohlcv

timestamp,open,high,low,close,volume
datetime[μs],f64,f64,f64,f64,f64
2025-07-26 08:55:00,0.999243,1.010463,0.999243,1.001332,596845.574424
2025-07-26 08:50:00,0.997004,1.004117,0.996041,0.999243,550992.664504
2025-07-26 08:45:00,0.999043,0.999524,0.99674,0.997004,137353.919695
2025-07-26 08:40:00,0.999257,0.999569,0.903255,0.999043,2.6478e6
2025-07-26 08:35:00,0.998526,0.999897,0.998513,0.999257,143837.742444
…,…,…,…,…,…
2025-07-26 01:00:00,0.997299,0.999389,0.992933,0.999389,502185.717434
2025-07-26 00:55:00,0.999449,1.003015,0.997299,0.997299,347396.981992
2025-07-26 00:50:00,1.000411,1.000658,0.99918,0.999449,71433.134071


In [31]:
validate_columns(df_ohlcv)

Invalid OHLC rows: False


#### Export to CSV / XLSX

In [58]:
df_ohlcv.write_csv("/home/vikas/Desktop/GitHub/CoinGecko_export_data/OHLCV_export.csv")
df_ohlcv.write_excel("/home/vikas/Desktop/GitHub/CoinGecko_export_data/OHLCV_export.xlsx",
                      autofit = True)